# Hidden State Extraction — Layer Sweep

Extracts hidden states from **all 32 transformer layers** of Llama-3.1-8B-Instruct using four extraction designs.
All 32 layers come for free from a single forward pass (`output_hidden_states=True`).

**Speed strategy:** 3 GPUs run in parallel via Python threads. Each GPU loads its own model copy
and processes 1/3 of conversations concurrently.

**Output per design:**
- `data/representations/{FOLDER_NAME}_layersweep_{X}/hidden_states.npy` — float16, shape `(N, 32, 4096)`
- `data/representations/{FOLDER_NAME}_layersweep_{X}/metadata.parquet` — one row per (conversation × turn)

Load one layer at a time in downstream notebooks using memory-mapped I/O:
```python
states = np.load(path, mmap_mode='r')   # does not load into RAM
layer_k = states[:, layer_idx, :]       # (N, 4096) — loads only this slice
```

**Designs**
| Letter | Extraction point | One fwd pass? |
|--------|-----------------|---------------|
| **D** | First token of each assistant response | Yes — 1 per conversation |
| **E** | Mean pool across all assistant response tokens | Yes — 1 per conversation |
| **F** | Last token of final assistant response, prior assistant responses masked as `.` | Yes — 1 per conversation |
| **B** | Last token of final assistant response, varying k prior turns (Bullwinkel design) | No — k passes per conversation |

Run Designs D, E, F first (fast). Run Design B only if needed — it is significantly slower.

Run this notebook once per folder. Change `FOLDER_NAME` in the config cell and re-run.

In [ ]:
import os
# ── Set visible GPUs before importing torch ───────────────────────────────────
# Adjust to the physical GPU IDs available on your node.
# Three IDs → three parallel workers. Use fewer if GPUs are limited.
GPU_IDS = [4, 5, 6]                    # ← physical GPU IDs on this node
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
# After setting CUDA_VISIBLE_DEVICES, logical device indices are 0, 1, 2 ...
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER_NAME = "actorattack_harmful"    # ← change to process a different folder

MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE      = torch.bfloat16

# Llama-3.1-8B has 32 transformer layers.
# hidden_states[0] = embedding; hidden_states[1..32] = transformer layers.
# Every 3rd layer: [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 32]
# Includes layer 32 explicitly so we always have the final layer for comparison.
N_TRANSFORMER_LAYERS = 32
LAYER_INDICES = list(range(3, N_TRANSFORMER_LAYERS, 3)) + [N_TRANSFORMER_LAYERS]
# → [3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 32]  (11 layers)

CONV_DIR  = repo_root / "data" / "conversations" / FOLDER_NAME
REPR_ROOT = repo_root / "data" / "representations"

parts     = FOLDER_NAME.split("_")
FRAMEWORK = parts[0]
SPLIT     = parts[1]

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"
files = sorted(CONV_DIR.glob("*.json"))

print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}")
print(f"Split:     {SPLIT}")
print(f"Input:     {CONV_DIR}")
print(f"Files:     {len(files)}")
print(f"GPUs:      {GPU_IDS} → logical {LOGICAL_GPU_IDS}")
print(f"Layers:    {LAYER_INDICES}  ({len(LAYER_INDICES)} total)")

In [ ]:
# ── Shared helpers (identical to 04b) ────────────────────────────────────────

def build_context(turns, framework):
    """Build filtered messages list. Skips is_refusal turns for Crescendo."""
    messages, extract_at = [], []
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t
    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_turn = pair.get("user")
        asst_turn = pair.get("assistant")
        if not user_turn or not asst_turn:
            continue
        if framework == "crescendo" and asst_turn.get("is_refusal", False):
            continue
        messages.append({"role": "user",      "content": user_turn["content"]})
        messages.append({"role": "assistant", "content": asst_turn["content"]})
        extract_at.append({"turn_idx": turn_idx, "n_messages": len(messages)})
    return messages, extract_at


def base_row(conv, fpath):
    return dict(
        framework = FRAMEWORK,
        split     = SPLIT,
        pair_id   = conv["objective_pair_id"],
        attempt   = conv.get("attempt", 1),
        verdict   = conv["verdict"],
        label     = 1 if SPLIT == "harmful" else 0,
        fname     = fpath.name,
    )


def get_turn_positions(tokenizer, messages, extract_at):
    """
    Precompute all token positions for a conversation in one pass over extract_at.
    Returns list of dicts with keys: first_asst, last_asst, last_user, asst_start, asst_end.
    All positions refer to indices in the full-conversation token sequence.
    """
    positions = []
    for entry in extract_at:
        n = entry["n_messages"]
        user_prefix = messages[:n - 1]   # up to (and including) user turn
        full_prefix = messages[:n]        # up to (and including) assistant turn

        gen_ids  = tokenizer.apply_chat_template(
            user_prefix, return_tensors="pt", add_generation_prompt=True)
        full_ids = tokenizer.apply_chat_template(
            full_prefix, return_tensors="pt", add_generation_prompt=False)
        user_ids = tokenizer.apply_chat_template(
            user_prefix, return_tensors="pt", add_generation_prompt=False)

        asst_start = gen_ids.shape[1]           # first assistant content token
        asst_end   = full_ids.shape[1]          # one past last assistant token
        last_user  = user_ids.shape[1] - 1      # last user message token
        first_asst = min(asst_start, asst_end - 1)
        last_asst  = asst_end - 1

        positions.append({
            "turn_idx":   entry["turn_idx"],
            "first_asst": first_asst,
            "last_asst":  last_asst,
            "last_user":  last_user,
            "asst_start": asst_start,
            "asst_end":   asst_end,
        })
    return positions


@torch.no_grad()
def forward_all_layers(model, input_ids):
    """
    Single forward pass. Returns a list of (seq_len, hidden_dim) tensors,
    one per LAYER_INDICES entry. Tensors are on CPU as float16.
    Frees GPU memory immediately after extraction.
    """
    outputs = model(input_ids, output_hidden_states=True)
    # hidden_states: tuple of length n_layers+1, each (1, seq_len, hidden_dim)
    layer_hiddens = [
        outputs.hidden_states[l][0].cpu().to(torch.float16)
        for l in LAYER_INDICES
    ]  # list of 32 tensors, each (seq_len, hidden_dim)
    del outputs
    return layer_hiddens  # list[32] of (seq_len, hidden_dim) cpu float16


def extract_point(layer_hiddens, pos):
    """Extract a single token position from all layers. Returns (n_layers, hidden_dim) float16."""
    return np.stack([lh[pos].numpy() for lh in layer_hiddens])  # (32, 4096)


def extract_meanpool(layer_hiddens, start, end):
    """Mean-pool a token range across all layers. Returns (n_layers, hidden_dim) float16."""
    if start >= end:
        end = start + 1
    return np.stack([lh[start:end].mean(dim=0).numpy() for lh in layer_hiddens])  # (32, 4096)


def load_done(save_dir):
    """Return set of (pair_id, attempt) already processed."""
    meta_path = save_dir / "metadata.parquet"
    if meta_path.exists():
        meta = pd.read_parquet(meta_path)
        done = set(zip(meta["pair_id"], meta["attempt"]))
        print(f"  Resuming: {len(done)} conversations already done")
        states = np.load(str(save_dir / "hidden_states.npy"))  # (N, 32, 4096)
        return done, [states], [meta]
    return set(), [], []


def save_design(states_list, meta_list, save_dir):
    """Concatenate and save. States shape: (N, n_layers, hidden_dim)."""
    save_dir.mkdir(parents=True, exist_ok=True)
    states = np.concatenate(states_list, axis=0).astype(np.float16)
    meta   = pd.concat(meta_list, ignore_index=True)
    np.save(str(save_dir / "hidden_states.npy"), states)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"  Saved {states.shape[0]} vectors → {save_dir}")
    print(f"  Shape: {states.shape}  ({states.nbytes / 1e9:.2f} GB)")
    return states, meta


print("Helpers defined.")

In [ ]:
# ── Parallel extraction infrastructure ───────────────────────────────────────

def run_parallel(files, extract_conv_fn, save_dir, done_set,
                 existing_states, existing_meta):
    pending = [
        f for f in files
        if (json.loads(f.read_text()).get("objective_pair_id"),
            json.loads(f.read_text()).get("attempt", 1)) not in done_set
    ]
    print(f"Pending: {len(pending)} / {len(files)} files")

    n_gpus  = len(LOGICAL_GPU_IDS)
    chunks  = [pending[i::n_gpus] for i in range(n_gpus)]

    all_states = list(existing_states)
    all_meta   = list(existing_meta)

    pbar = tqdm(total=len(pending), desc="Extracting")
    lock = threading.Lock()

    def worker(gpu_id, chunk):
        torch.cuda.set_device(gpu_id)
        device = f"cuda:{gpu_id}"
        tqdm.write(f"GPU {gpu_id}: loading model...")
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=DTYPE, device_map={"": device}
        )
        mdl.eval()
        tqdm.write(f"GPU {gpu_id}: model ready, processing {len(chunk)} files")

        local_states, local_meta = [], []

        for fpath in chunk:
            conv = json.loads(fpath.read_text())
            try:
                states, rows = extract_conv_fn(mdl, tok, conv, fpath)
                if states is not None and len(rows) > 0:
                    local_states.append(states)
                    local_meta.append(pd.DataFrame(rows))
            except Exception as e:
                tqdm.write(f"GPU {gpu_id} ERROR {fpath.name}: {e}")
            with lock:
                pbar.update(1)

        tqdm.write(f"GPU {gpu_id}: done")
        with lock:
            all_states.extend(local_states)
            all_meta.extend(local_meta)

        del mdl
        torch.cuda.empty_cache()

    threads = [
        threading.Thread(target=worker, args=(gpu_id, chunks[gpu_id]))
        for gpu_id in LOGICAL_GPU_IDS
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    pbar.close()
    return all_states, all_meta


print("Parallel infrastructure defined.")

## Design D — First token of each assistant response

Extracts at the first generated token of each assistant response — the model's sharpest decision point.
One forward pass per conversation; 6 extraction positions (one per turn).

**Research question:** At which layer does the model make its compliance decision?

In [ ]:
SAVE_D = REPR_ROOT / f"{FOLDER_NAME}_layersweep_D"
done_d, existing_states_d, existing_meta_d = load_done(SAVE_D)

def extract_design_D(model, tokenizer, conv, fpath):
    turns = conv.get("turns", [])
    if not turns:
        return None, []
    messages, extract_at = build_context(turns, FRAMEWORK)
    if not extract_at:
        return None, []

    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=False
    ).to(model.device)

    layer_hiddens = forward_all_layers(model, input_ids)
    positions     = get_turn_positions(tokenizer, messages, extract_at)

    vecs = np.stack([extract_point(layer_hiddens, p["first_asst"]) for p in positions])
    # vecs: (n_turns, 32, 4096)

    rows = [
        {**base_row(conv, fpath), "turn_idx": p["turn_idx"]}
        for p in positions
    ]
    return vecs, rows


states_d, meta_d = run_parallel(
    files, extract_design_D, SAVE_D, done_d, existing_states_d, existing_meta_d
)
save_design(states_d, meta_d, SAVE_D)

## Design E — Mean pool across all assistant response tokens

Averages all hidden states across the full assistant response at each turn.
Integrates the sustained compliance/refusal signal across all generated tokens.
One forward pass per conversation.

**Research question:** At which layer is the distributed compliance signal strongest?

In [ ]:
SAVE_E = REPR_ROOT / f"{FOLDER_NAME}_layersweep_E"
done_e, existing_states_e, existing_meta_e = load_done(SAVE_E)

def extract_design_E(model, tokenizer, conv, fpath):
    turns = conv.get("turns", [])
    if not turns:
        return None, []
    messages, extract_at = build_context(turns, FRAMEWORK)
    if not extract_at:
        return None, []

    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=False
    ).to(model.device)

    layer_hiddens = forward_all_layers(model, input_ids)
    positions     = get_turn_positions(tokenizer, messages, extract_at)

    vecs = np.stack([
        extract_meanpool(layer_hiddens, p["asst_start"], p["asst_end"])
        for p in positions
    ])  # (n_turns, 32, 4096)

    rows = [
        {**base_row(conv, fpath), "turn_idx": p["turn_idx"]}
        for p in positions
    ]
    return vecs, rows


states_e, meta_e = run_parallel(
    files, extract_design_E, SAVE_E, done_e, existing_states_e, existing_meta_e
)
save_design(states_e, meta_e, SAVE_E)

## Design F — Final turn, prior assistant responses masked

Prior assistant responses replaced with `.`. Extracts at the last token of the final assistant response.
Isolates the intrinsic compliance signal from the contextual warm-up effect.
One forward pass per conversation; one extraction point (final turn only).

**Research question:** At which layer does the contextual warm-up effect appear?
Compare F vs E across layers: where they diverge is where prior context contributes.

In [ ]:
SAVE_F = REPR_ROOT / f"{FOLDER_NAME}_layersweep_F"
done_f, existing_states_f, existing_meta_f = load_done(SAVE_F)

PLACEHOLDER = "."

def mask_prior_assistant(messages):
    asst_positions = [i for i, m in enumerate(messages) if m["role"] == "assistant"]
    last_asst_idx  = asst_positions[-1] if asst_positions else -1
    return [
        ({"role": "assistant", "content": PLACEHOLDER}
         if m["role"] == "assistant" and i != last_asst_idx
         else m)
        for i, m in enumerate(messages)
    ]


def extract_design_F(model, tokenizer, conv, fpath):
    turns = conv.get("turns", [])
    if not turns:
        return None, []
    messages, extract_at = build_context(turns, FRAMEWORK)
    if not extract_at:
        return None, []

    masked_messages = mask_prior_assistant(messages)

    input_ids = tokenizer.apply_chat_template(
        masked_messages, return_tensors="pt", add_generation_prompt=False
    ).to(model.device)

    layer_hiddens = forward_all_layers(model, input_ids)

    # Extract at last token of final assistant response
    pos = input_ids.shape[1] - 1
    vec = extract_point(layer_hiddens, pos)   # (32, 4096)

    final_entry = extract_at[-1]
    rows = [{
        **base_row(conv, fpath),
        "turn_idx": final_entry["turn_idx"],
        "n_turns":  len(extract_at),
    }]
    return vec[np.newaxis, :, :], rows   # (1, 32, 4096)


states_f, meta_f = run_parallel(
    files, extract_design_F, SAVE_F, done_f, existing_states_f, existing_meta_f
)
save_design(states_f, meta_f, SAVE_F)

## Design B — Varying context depth (Bullwinkel design) *(optional — slower)*

Extracts at the last token of the **final** assistant response with k = 1..n_turns prior turns in context.
Requires k forward passes per conversation — significantly slower than D/E/F.

**Research question:** At which layer does adding more prior context increase compliance signal?

In [ ]:
SAVE_B = REPR_ROOT / f"{FOLDER_NAME}_layersweep_B"

meta_path_b = SAVE_B / "metadata.parquet"
if meta_path_b.exists():
    existing_b_meta   = pd.read_parquet(meta_path_b)
    done_b_keys       = set(zip(existing_b_meta["pair_id"],
                                existing_b_meta["attempt"],
                                existing_b_meta["k"]))
    existing_states_b = [np.load(str(SAVE_B / "hidden_states.npy"))]
    existing_meta_b   = [existing_b_meta]
    print(f"  Resuming: {len(done_b_keys)} (pair_id, attempt, k) triples already done")
else:
    done_b_keys       = set()
    existing_states_b = []
    existing_meta_b   = []

results_lock_b = threading.Lock()
all_states_b   = list(existing_states_b)
all_meta_b     = list(existing_meta_b)

# Filter to conversations not fully done
pending_b = []
for f in files:
    conv    = json.loads(f.read_text())
    pair_id = conv["objective_pair_id"]
    attempt = conv.get("attempt", 1)
    turns   = conv.get("turns", [])
    messages, extract_at = build_context(turns, FRAMEWORK)
    if not extract_at:
        continue
    n_turns = len(messages) // 2
    if any((pair_id, attempt, k) not in done_b_keys for k in range(1, n_turns + 1)):
        pending_b.append(f)

print(f"Pending conversations for Design B: {len(pending_b)}")

pbar_b = tqdm(total=len(pending_b), desc="Design B")

def worker_B(gpu_id, chunk):
    torch.cuda.set_device(gpu_id)
    device = f"cuda:{gpu_id}"
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, device_map={"": device}
    )
    mdl.eval()

    local_states, local_meta = [], []

    for fpath in chunk:
        conv    = json.loads(fpath.read_text())
        pair_id = conv["objective_pair_id"]
        attempt = conv.get("attempt", 1)
        turns   = conv.get("turns", [])
        if not turns:
            with results_lock_b:
                pbar_b.update(1)
            continue

        try:
            messages, extract_at = build_context(turns, FRAMEWORK)
            if not extract_at:
                continue
            n_turns = len(messages) // 2

            for k in range(1, n_turns + 1):
                if (pair_id, attempt, k) in done_b_keys:
                    continue
                context   = messages[-(2 * k):]
                input_ids = tok.apply_chat_template(
                    context, return_tensors="pt", add_generation_prompt=False
                ).to(device)
                layer_hiddens = forward_all_layers(mdl, input_ids)
                pos = input_ids.shape[1] - 1
                vec = extract_point(layer_hiddens, pos)
                local_states.append(vec[np.newaxis, :, :])
                local_meta.append(pd.DataFrame([{
                    **base_row(conv, fpath),
                    "k": k, "n_turns": n_turns,
                    "turn_idx": extract_at[-1]["turn_idx"],
                }]))
        except Exception as e:
            tqdm.write(f"GPU {gpu_id} ERROR {fpath.name}: {e}")

        with results_lock_b:
            pbar_b.update(1)

    with results_lock_b:
        all_states_b.extend(local_states)
        all_meta_b.extend(local_meta)

    del mdl
    torch.cuda.empty_cache()


n_gpus   = len(LOGICAL_GPU_IDS)
chunks_b = [pending_b[i::n_gpus] for i in range(n_gpus)]

threads_b = [
    threading.Thread(target=worker_B, args=(gpu_id, chunks_b[gpu_id]))
    for gpu_id in LOGICAL_GPU_IDS
]
for t in threads_b:
    t.start()
for t in threads_b:
    t.join()

pbar_b.close()
save_design(all_states_b, all_meta_b, SAVE_B)

## Verification

In [ ]:
for design_letter, save_dir in [("D", SAVE_D), ("E", SAVE_E), ("F", SAVE_F)]:
    states_path = save_dir / "hidden_states.npy"
    meta_path   = save_dir / "metadata.parquet"
    if not states_path.exists():
        print(f"Design {design_letter}: not found")
        continue

    # Memory-map to avoid loading all 3 layers × full array into RAM
    states = np.load(str(states_path), mmap_mode="r")
    meta   = pd.read_parquet(meta_path)

    print(f"Design {design_letter}:")
    print(f"  Shape:  {states.shape}  ({states.nbytes / 1e9:.2f} GB)")
    print(f"  Rows:   {len(meta)}")
    print(f"  Verdicts: {dict(meta['verdict'].value_counts())}")

    # Sanity check last layer matches expected stats
    sample = states[:min(200, len(states)), -1, :].astype(np.float32)
    print(f"  Layer 32 stats: mean={sample.mean():.4f}  std={sample.std():.4f}  "
          f"NaNs={np.isnan(sample).sum()}  Infs={np.isinf(sample).sum()}")
    print()